# Explore Zenodo Recipe Dataset

**Mission**: Data exploratory on the Zenodo recipe dataset stored in `/workspace/ICOCO.duckdb`.

**Goal**: Answer 5 counts (with SQL-first approach):
1. How many dishes?
2. How many dishes with labels / without labels?
3. How many unique labels (element level)?
4. How many unique ingredients (element level)?
5. How many cooking methods (element level)?

The relevant table is the silver layer **`silver.RecipeZenodo`** created in `1_silver_create/silver_curation.ipynb`.

**Principles**
- **SQL-first** — every question answered with DuckDB SQL.
- **Read-only** — this notebook only *reads*; no tables are created or modified.
- **Element level** — columns `Category`, `Cuisine`, `Ingredients`, `Cooking Methods` are arrays (`VARCHAR[]`); we `UNNEST` them to count each element individually.


## 0. Setup (connect to DuckDB)

Follows the same connection pattern as the silver notebook. Only **one** DuckDB connection is open at a time (DuckDB limitation).

In [1]:
%load_ext sql
%config SqlMagic.displaylimit = 100  # show all rows in query results
%sql duckdb:////workspace/ICOCO.duckdb

Connecting to 'duckdb:////workspace/ICOCO.duckdb'

In [2]:
# Close the connection first to flush the WAL file (same pattern as silver notebook).
%sql --close duckdb:////workspace/ICOCO.duckdb

In [3]:
# Reconnect with a fresh connection (recovers/checkpoints the WAL).
%sql duckdb:////workspace/ICOCO.duckdb

Connecting and switching to connection 'duckdb:////workspace/ICOCO.duckdb'

## 0.1 Table overview

`schema` of `silver.RecipeZenodo` plus a peek at a few rows to see what the data looks like.

In [4]:
%%sql
DESCRIBE silver.RecipeZenodo;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

column_name,column_type,null,key,default,extra


In [5]:
%%sql
SELECT Name, Category, Cuisine, Ingredients, "Cooking Methods"
FROM silver.RecipeZenodo
LIMIT 3;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Name,Category,Cuisine,Ingredients,Cooking Methods
Pineapple Glaze for Ham,['Dinner'],['American'],"['pineapple', 'maraschino cherries', 'brown sugar']",['microwave']
Awesome Egg Rolls,"['Dinner', 'Appetizer']","['World', 'Asian']","['vegetable oil', 'egg', 'cabbage', 'bean sprouts', 'carrot', 'stalk', 'onion', 'shrimp', 'soy sauce', 'garlic powder', 'pepper to taste', 'cornstarch', 'oil for frying', 'water', 'roll wrappers']",['fry']
Spanish Tortilla,['Breakfast'],['Spanish'],"['olive oil', 'potatoes', 'bacon', 'ham', 'onion', 'bell pepper', 'eggs', 'spanish seasoning']",['fry']


## 1. How many dishes?

A **dish** = one row in `silver.RecipeZenodo`. Total row count answers the question.

In [6]:
%%sql
SELECT COUNT(*) AS n_dishes
FROM silver.RecipeZenodo;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

n_dishes
50511


## About labels (Cuisine) — the list in each dish

The mission note says *“labels of a dish is a list such as ["America", "Mexico"]”* — these are **cuisine/country** labels.

They are stored in the `Cuisine` array column, e.g.:

In [7]:
%%sql
SELECT Name, Cuisine
FROM silver.RecipeZenodo
LIMIT 5;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Name,Cuisine
Pineapple Glaze for Ham,['American']
Awesome Egg Rolls,"['World', 'Asian']"
Spanish Tortilla,['Spanish']
Spicy Tarragon Yogurt Chicken,['American']
Apple Bread,['American']


## 2. How many dishes with / without labels?

A dish **has a label** when its `Cuisine` array is non-empty; otherwise it is unlabelled.

(`len([]) = 0` is used because every row is non-NULL.)

In [8]:
%%sql
SELECT
  CASE WHEN len(Cuisine) > 0 THEN 'with label' ELSE 'without label' END AS label_status,
  COUNT(*) AS n_dishes
FROM silver.RecipeZenodo
GROUP BY label_status
ORDER BY label_status DESC;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

label_status,n_dishes
without label,18926
with label,31585


## 3. How many unique labels (element level)?

Here we **flatten** each dish's `Cuisine` array with `UNNEST` (one row per label element), then count distinct values.

In [9]:
%%sql
SELECT COUNT(DISTINCT e) AS unique_labels
FROM silver.RecipeZenodo, UNNEST(Cuisine) AS t(e);

Running query in 'duckdb:////workspace/ICOCO.duckdb'

unique_labels
116


## 4. How many unique ingredients (element level)?

Same `UNNEST` technique over `Ingredients` (a `VARCHAR[]` of ingredient names).

In [10]:
%%sql
SELECT COUNT(DISTINCT e) AS unique_ingredients
FROM silver.RecipeZenodo, UNNEST(Ingredients) AS t(e);

Running query in 'duckdb:////workspace/ICOCO.duckdb'

unique_ingredients
17033


## 5. How many cooking methods (element level)?

Same again over `Cooking Methods` (`VARCHAR[]`). The column name contains a space, so it is double-quoted.

In [11]:
%%sql
SELECT COUNT(DISTINCT e) AS unique_cooking_methods
FROM silver.RecipeZenodo, UNNEST("Cooking Methods") AS t(e);

Running query in 'duckdb:////workspace/ICOCO.duckdb'

unique_cooking_methods
22


## Summary

| Question | Answer |
|----------|--------|
| 1. Number of dishes | **50,511** |
| 2. Dishes with label / without label | **31,585** / **18,926** |
| 3. Unique labels (Cuisine, element level) | **116** |
| 4. Unique ingredients (element level) | **17,033** |
| 5. Unique cooking methods (element level) | **22** |

**Robustness note**: the discrete counts above compare *exact* element strings. If we normalise whitespace and case (`lower(trim(e))`):
- Cuisine: **116** (unchanged)
- Ingredients: **17,033 → 17,014** (19 variants differ only by spacing/case, e.g. `tomato` vs `tomato `)
- Cooking methods: **22** (unchanged)

### Top labels for context


In [12]:
%%sql
SELECT e AS label, COUNT(*) AS n_dishes
FROM silver.RecipeZenodo, UNNEST(Cuisine) AS t(e)
GROUP BY e
ORDER BY n_dishes DESC
LIMIT 8;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

label,n_dishes
American,17664
U.S.,1789
Mexican Inspired,1499
Italian,1475
Italian Inspired,1317
Mexican,1049
Southern,973
Asian Inspired,897
